In [ ]:
import logging

logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
)
logging.getLogger("sum_nsm_dynamic_pricing.src.sum_nsm_dynamic_pricing.routing.route_builder").setLevel(logging.DEBUG)


In [1]:
from pathlib import Path
import time

from sum_nsm_dynamic_pricing.src.sum_nsm_dynamic_pricing import InstancePipeline, InstanceParams, PipelineConfig
from sum_nsm_dynamic_pricing.src.sum_nsm_dynamic_pricing.stations.providers import DonkeyRepublicProvider
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
)
logging.getLogger("sum_nsm_dynamic_pricing.src.sum_nsm_dynamic_pricing.routing.route_builder").setLevel(logging.INFO)


def main() -> None:
    total_start = time.perf_counter()

    config = PipelineConfig(
        center_lat=46.2059149,
        center_lon=6.1438511,
        radius_m=1000,
        output_dir=Path("data/sum_nsm_dynamic_pricing/geneva_1km"),
        gtfs_path=Path("/Users/rebecamurillo/Documents/INRIA/development/inocs-sum-gtfs-geojson/otp/data/geneva-2026-gtfs.zip"),
        gbfs_path=None,
        seed=42,
        # min_od_distance_m=100
    )
    instance_params = InstanceParams(
        # num_time_steps=24,
        # time_step_duration_minutes=60,
        # num_agents=1000,
        # demand_distribution="uniform",
        # price_sensitivity=0.5,
    )

    pipeline = InstancePipeline(config=config, gbfs_provider=DonkeyRepublicProvider())

    start = time.perf_counter()
    stations_path = pipeline.prepare_stations()
    print(f"[TIMING] prepare_stations: {time.perf_counter() - start:.2f}s")

    start = time.perf_counter()
    hubs_path = pipeline.build_hubs(stations_path)
    print(f"[TIMING] build_hubs: {time.perf_counter() - start:.2f}s")

    start = time.perf_counter()
    routes_path = pipeline.compute_routes(hubs_path)
    print(f"[TIMING] compute_routes: {time.perf_counter() - start:.2f}s")
    print(f"Starting route quality validation for: {routes_path}")
    validated = pipeline.quality_routes(routes_path)
    print(f"Completed route quality validation for: {routes_path} - Validated: {validated}")

    start = time.perf_counter()
    instance_path = pipeline.generate_instance(instance_params)
    print(f"[TIMING] generate_instance: {time.perf_counter() - start:.2f}s")

    print(f"Instance generated at: {instance_path}")
    print(f"[TIMING] total: {time.perf_counter() - total_start:.2f}s")


if __name__ == "__main__":
    main()


[TIMING] prepare_stations: 0.42s


2026-03-12 17:22:10,565 [INFO] pyogrio._io: Created 33 records
2026-03-12 17:22:10,569 [INFO] pyogrio._io: Created 339 records
2026-03-12 17:22:10,579 [INFO] pyogrio._io: Created 33 records


[TIMING] build_hubs: 2.73s


2026-03-12 17:23:07,389 [INFO] sum_nsm_dynamic_pricing.src.sum_nsm_dynamic_pricing.routing.route_builder: Route building complete: 862/862 OD pairs returned results. Output: data/sum_nsm_dynamic_pricing/geneva_1km/od_routes.json
2026-03-12 17:23:08,443 [ERROR] sum_nsm_dynamic_pricing.src.sum_nsm_dynamic_pricing.quality: OD 0|7 has no PT/Bike options (arc_pt_bike + arc_pt_only are empty)
2026-03-12 17:23:08,443 [ERROR] sum_nsm_dynamic_pricing.src.sum_nsm_dynamic_pricing.quality: OD 0|16 has no PT/Bike options (arc_pt_bike + arc_pt_only are empty)
2026-03-12 17:23:08,444 [ERROR] sum_nsm_dynamic_pricing.src.sum_nsm_dynamic_pricing.quality: OD 0|18 has no car options (arc_car is empty)
2026-03-12 17:23:08,444 [ERROR] sum_nsm_dynamic_pricing.src.sum_nsm_dynamic_pricing.quality: OD 0|6 has no car options (arc_car is empty)
2026-03-12 17:23:08,444 [ERROR] sum_nsm_dynamic_pricing.src.sum_nsm_dynamic_pricing.quality: OD 0|20 has no PT/Bike options (arc_pt_bike + arc_pt_only are empty)
2026-03-1

[TIMING] compute_routes: 57.79s
Starting route quality validation for: data/sum_nsm_dynamic_pricing/geneva_1km/od_routes.json


2026-03-12 17:23:09,120 [INFO] sum_nsm_dynamic_pricing.src.sum_nsm_dynamic_pricing.quality: OD cleanup removed 252/862 OD pairs (29.23% invalid) from data/sum_nsm_dynamic_pricing/geneva_1km/od_routes.json
2026-03-12 17:23:09,182 [INFO] sum_nsm_dynamic_pricing.src.sum_nsm_dynamic_pricing.quality: Route quality validation passed for data/sum_nsm_dynamic_pricing/geneva_1km/od_routes.json (610 OD pairs)


Completed route quality validation for: data/sum_nsm_dynamic_pricing/geneva_1km/od_routes.json - Validated: True


2026-03-12 17:23:09,963 [ERROR] sum_nsm_dynamic_pricing.src.sum_nsm_dynamic_pricing.quality: delta_ia for arc '5506' is unbalanced (sum=1.0, expected 0)
2026-03-12 17:23:09,964 [ERROR] sum_nsm_dynamic_pricing.src.sum_nsm_dynamic_pricing.quality: delta_ia for arc '5578' is unbalanced (sum=1.0, expected 0)
2026-03-12 17:23:09,964 [ERROR] sum_nsm_dynamic_pricing.src.sum_nsm_dynamic_pricing.quality: Instance quality validation failed for data/sum_nsm_dynamic_pricing/geneva_1km/Instance_610.json
2026-03-12 17:23:09,969 [WARNING] sum_nsm_dynamic_pricing.src.sum_nsm_dynamic_pricing.pipeline: Instance quality validation reported issues. See error logs for details.


[TIMING] generate_instance: 0.78s
Instance generated at: data/sum_nsm_dynamic_pricing/geneva_1km/Instance_610.json
[TIMING] total: 62.55s


## Optimization and Metrics Step
Run the exact optimization model on the generated instance and then export summary metrics.

In [1]:
from pathlib import Path

from sum_nsm_dynamic_pricing.src.sum_nsm_dynamic_pricing import (
    InstancePipeline,
    OptimizerParams,
    PipelineConfig,
    run_pipeline,
    InstanceParams,
 )
from sum_nsm_dynamic_pricing.src.sum_nsm_dynamic_pricing.stations.providers import DonkeyRepublicProvider

# Reuse your previous output folder. Change to geneva_1km if needed.
# output_dir = Path("data/sum_nsm_dynamic_pricing/geneva_2km")
output_dir = Path("data/sum_nsm_dynamic_pricing/geneva_1km")

config = PipelineConfig(
    center_lat=46.2059149,
    center_lon=6.1438511,
    radius_m=1000,
    output_dir=output_dir,
    gtfs_path=Path("/Users/rebecamurillo/Documents/INRIA/development/inocs-sum-gtfs-geojson/otp/data/geneva-2026-gtfs.zip"),
    gbfs_path=None,
    seed=42,
 )

pipeline = InstancePipeline(config=config, gbfs_provider=DonkeyRepublicProvider())

# Optional: run the full instance generation pipeline if no instance exists yet.
instance_candidates = sorted(output_dir.glob("Instance_*.json"))
if instance_candidates:
    instance_path = instance_candidates[-1]
else:
    instance_path = "NONE"
    # instance_path = run_pipeline(config, InstanceParams(), gbfs_provider=DonkeyRepublicProvider())

print(f"Using instance: {instance_path}")

params = OptimizerParams(
        max_displayed_alternatives=10,
        P_bar=0.6,
        pt_price=2.0,
        cost_bike=0.6,
        parking_fee=2.8,
        theta=0.5,
        ridership_weight=8.0,
        verbose=False,
        verbose2=False,
    )
solution_path = pipeline.optimize(
    params=params,
    instance_path=instance_path,
 )

# solution_path = output_dir / "optimization_instance_results.json"
print(f"Optimization results: {solution_path}")

metrics_path = pipeline.summarize_optimization(solution_path, params)
print(f"Optimization metrics: {metrics_path}")

Using instance: data/sum_nsm_dynamic_pricing/geneva_1km/Instance_610.json
⚠️ Dropped 45 stations not used in delta_ia: ['bs_18394', 'bs_18471', 'bs_18497', 'bs_18503', 'bs_18509', 'bs_18520', 'bs_18538', 'bs_18555', 'bs_18565', 'bs_18570', 'bs_18589', 'bs_18593', 'bs_18611', 'bs_18807', 'bs_18835', 'bs_18837', 'bs_18840', 'bs_18841', 'bs_18844', 'bs_20626', 'bs_20712', 'bs_20714', 'bs_20720', 'bs_21902', 'bs_21905', 'bs_21906', 'bs_21909', 'bs_21938', 'bs_21940', 'bs_21960', 'bs_21965', 'bs_21966', 'bs_21969', 'bs_22022', 'bs_22025', 'bs_22102', 'bs_22185', 'bs_22187', 'bs_22189', 'bs_22213', 'bs_22237', 'bs_22363', 'bs_22365', 'bs_22414', 'bs_27876']
Set parameter Username
Set parameter LicenseID to value 2790753
Academic license - for non-commercial use only - expires 2027-03-12
Set parameter NodefileStart to value 2
Set parameter NodefileDir to value "C:\b_b_gurobi"
Set parameter Threads to value 4
Set parameter Method to value -1
Set parameter TimeLimit to value 7200
Starting optim